# 18 · Flag diffs that break your team's rules

## The problem

Every codebase has rules a linter can't express: *"all database access goes through the repository layer"*, *"never log
request bodies"*, *"every new endpoint needs auth"*. They live in a wiki page nobody reads. Reviewers forget them under
time pressure, and new hires don't know they exist, so the same comment gets typed by hand on PR after PR.

## Why Jev

Writing a custom lint rule for each of these is a lot of AST work, and asking an LLM to "review this diff" gives you a
wall of prose with a different opinion every run. What you actually want is **one yes/no per rule**, with a probability
you can threshold, cheap enough to run on every push. That is a System One job: Jev answers typed questions with
**calibrated probabilities** in about 100 ms, for a fraction of a cent, and all the rules go in **one call** per diff.

It's a **review signal**, never a replacement for tests, static analysis or a human reviewer.

## What we'll build

```
diff ──► Jev: one Noul per team rule, ONE call ──► p ≥ COMMENT_AT? ──yes──► post a review comment
          • no_raw_sql_in_handlers                        │                   + count it in `fired`
          • no_logging_bodies                             no
          • auth_on_new_endpoints                         ▼
          • i18n_strings                                stay quiet
```

## What you'll learn
- **Natural-language policy as code**: rules written in English, checked by a `Noul` each.
- How `state` can be a **dict**, and how backtick names in the instructions (`` `diff` ``) point at its keys.
- Why a **quiet** bot matters as much as an accurate one, and how a per-rule **fire count** tells you which rule is noisy.

*From the use-case catalog: [18 practical Jev use cases for AI agents](https://huggingface.co/blog/karmen-beatapi/18-practical-jev-use-cases-for-ai-agents), #12.*

> **How to run this notebook**
> 1. Pick the **`.venv`** kernel (top right in VS Code, *Kernel → Change kernel* in Jupyter).
> 2. Run the cells **top to bottom** (or *Run All*). Every cell builds on the ones above it.
> 3. If you see `NameError: name '...' is not defined`, the kernel was restarted (switching kernels does this) and forgot the earlier cells. Click **Run All** or *Run All Above*, then continue.

## Step 0 · Setup

This cell:
- loads the Jev API key (`TYPESAFE_API_KEY`) from the `.env` file at the repo root,
- creates the `jev` client (one object, reused for every call),
- imports the question types. In this notebook we use **`Noul`** (yes/no). `NoulCriteria`, `Score` and `Choice` are imported for the exercises,
- defines `bar()`, a tiny helper that draws a 0..1 probability as a text bar so the outputs are easy to read at a glance.

In [1]:
import os, re
from pathlib import Path
from dotenv import load_dotenv
from typesafe_sdk import TypeSafeClient, Noul, NoulCriteria, Score, Choice

for folder in [Path.cwd(), *Path.cwd().parents]:
    if (folder / ".env").exists():
        load_dotenv(folder / ".env")
        break

jev = TypeSafeClient()


def bar(p, width=20):
    """A unicode progress bar for a 0..1 value."""
    filled = round(max(0.0, min(1.0, p)) * width)
    return "█" * filled + "·" * (width - filled)


print("Jev client ready")

Jev client ready


## Step 1 · The team's rules, in English

Each rule is one sentence a human reviewer would agree with. `rule_questions()` wraps every rule in a `Noul`:

- **How a `Noul` works:** it's a yes/no question and `.noul` is the **probability that the answer is yes** (0 to 1). Here
  "yes" means *this diff violates the rule*.
- **`state` as a dict:** we'll send the diff as `state={"diff": ...}`. In the instructions, `` `diff` `` in backticks refers to
  that key, so Jev knows exactly which text the question is about. The instruction also says to judge only the **added**
  lines (starting with `+`): removed code is not this PR's fault.

This cell only **defines** the rules. Nothing is sent to Jev yet.

> **Workshop note:** two rules were reworded. Originally `auth_on_new_endpoints` and `i18n_strings` scored only around
> 0.65 on a diff that clearly broke them, uncomfortably close to the comment threshold. Spelling out what counts
> (*"a new `@app` route"*, *"e.g. an `@require_auth` decorator"*, *"messages, titles or labels shown to users"*) pushed real
> violations clearly above it and kept clean diffs near 0. Concrete examples in a rule are as useful to the model as they
> are to a new hire.

In [2]:
RULES = {
    "no_raw_sql_in_handlers": "HTTP handlers must not contain raw SQL; database access goes through the repository layer",
    "no_logging_bodies": "Never log full request or response bodies, since they may contain personal data",
    "auth_on_new_endpoints": "Every new HTTP endpoint (a new @app route) must require authentication, e.g. with an @require_auth decorator",
    "i18n_strings": "User-facing strings (messages, titles or labels shown to users) must be wrapped in the t() translation function, e.g. t(\"orders.started\")",
}


def rule_questions():
    return {k: Noul(instructions=f"Does the ADDED code (lines starting with +) in `diff` violate this rule: {rule}?")
            for k, rule in RULES.items()}


for k in RULES:
    print("-", k)

- no_raw_sql_in_handlers
- no_logging_bodies
- auth_on_new_endpoints
- i18n_strings


## Step 2 · Look at one raw answer

`DIFF_1` is a deliberately bad new endpoint. Read it and count the violations yourself before running the next cell:
it logs the request body, builds raw SQL inside the handler, has no auth decorator, and returns an untranslated
`"Export started!"` message.

This cell defines the diff only. The call is in the cell after it.

In [3]:
DIFF_1 = """
+@app.post("/orders/export")
+def export_orders(request):
+    logger.info("export request: %s", request.body)
+    rows = db.execute("SELECT * FROM orders WHERE user_id = %s" % request.user_id)
+    return {"message": "Export started!", "rows": rows}
"""
print(DIFF_1)


+@app.post("/orders/export")
+def export_orders(request):
+    logger.info("export request: %s", request.body)
+    rows = db.execute("SELECT * FROM orders WHERE user_id = %s" % request.user_id)
+    return {"message": "Export started!", "rows": rows}



Now one `system_one` call with **all four rules**, printed as bars.

In [4]:
r = jev.system_one(state={"diff": DIFF_1}, questions=rule_questions())
for k in RULES:
    print(f"  {k:<24} {bar(r.nouls[k].noul)} {r.nouls[k].noul:.2f}")

  no_raw_sql_in_handlers   ███████████████████· 0.97
  no_logging_bodies        ███████████████████· 0.94
  auth_on_new_endpoints    ███████████████████· 0.93
  i18n_strings             ████████████████···· 0.82


**What to notice**
- All four rules come back **high**: Jev found every violation we counted by eye.
- All four sit well clear of the 0.60 comment line we'll use next. `i18n_strings` tends to be the least certain, since deciding whether a string is *user-facing* takes a little inference.
- Four independent yes/no answers, **one** request.

## Step 3 · Turn probabilities into review comments

`review(diff, client)` posts a comment only for rules at or above `COMMENT_AT`. Everything below stays silent.

**The fire count.** A review bot dies when people learn to ignore it. So every time a rule fires we add one to
`fired[rule]` (a `Counter`). Over a week of PRs, divide each rule's count by the number of diffs reviewed:

| fire rate for a rule | what it usually means | what to do |
|---|---|---|
| near 0 | rule is rarely broken, or worded too narrowly | spot-check a few PRs by hand |
| a few % | it's catching real problems | keep it |
| a large share of all diffs | it's **noisy**: vague wording, or it fires on code it shouldn't | rewrite the rule, or raise its threshold |

This cell only defines the function and the counters.

In [5]:
from collections import Counter

COMMENT_AT = 0.60
fired = Counter()      # rule -> how many diffs it commented on
reviewed = 0           # how many diffs we've reviewed


def review(diff, client):
    global reviewed
    if "RULES" not in globals():
        raise RuntimeError("RULES is missing: the kernel restarted. Use 'Run All' (or run the cells above) first.")

    r = client.system_one(state={"diff": diff}, questions=rule_questions())
    reviewed += 1
    comments = []
    for k, rule in RULES.items():
        p = r.nouls[k].noul
        if p >= COMMENT_AT:
            fired[k] += 1
            comments.append((k, p, rule))
    return comments


print("review() ready, comment threshold", COMMENT_AT)

review() ready, comment threshold 0.6


## Step 4 · Review four diffs

Besides `DIFF_1` we add three more, chosen to test both directions:

| Diff | What it is | Comments we want |
|---|---|---|
| `DIFF_1` | the bad export endpoint from Step 2 | all four |
| `DIFF_2` | a new `/health` endpoint **with** `@require_auth` and a `t()` string | **none** (a clean diff must stay quiet) |
| `DIFF_3` | a new `/admin/users` endpoint, **no auth**, and a hard-coded `title="All users"` | `auth_on_new_endpoints` and `i18n_strings` only (it uses the repository, so no SQL comment) |
| `DIFF_4` | a small refactor, not an endpoint at all | **none** |

This cell defines the diffs; the next one reviews them.

In [6]:
DIFF_2 = """
+@app.get("/health")
+@require_auth
+def health(request):
+    return {"status": t("service.healthy")}
"""
DIFF_3 = """
+@app.get("/admin/users")
+def list_users(request):
+    users = user_repo.list_all()
+    return render("users.html", title="All users", users=users)
"""
DIFF_4 = """
-def total(items):
-    return sum([i.price * i.qty for i in items])
+def total(items):
+    return sum(i.price * i.qty for i in items)
"""
DIFFS = {"DIFF_1": DIFF_1, "DIFF_2": DIFF_2, "DIFF_3": DIFF_3, "DIFF_4": DIFF_4}
print(len(DIFFS), "diffs ready")

4 diffs ready


We reset the counters first, so re-running this cell always gives the same tally. Then each diff is reviewed, and at the
end we print each rule's **fire count** and **fire rate** across the batch.

In [7]:
fired.clear()
reviewed = 0

for name, diff in DIFFS.items():
    comments = review(diff, jev)
    print(f"{name}: {len(comments)} comment(s)")
    for k, p, rule in comments:
        print(f"   ⚠ {k} ({p:.2f}): {rule}")

print(f"\nrule fire counts over {reviewed} diffs:")
for k in RULES:
    print(f"   {k:<24} {fired[k]}  ({fired[k] / reviewed:.0%})")

DIFF_1: 4 comment(s)
   ⚠ no_raw_sql_in_handlers (0.97): HTTP handlers must not contain raw SQL; database access goes through the repository layer
   ⚠ no_logging_bodies (0.92): Never log full request or response bodies, since they may contain personal data
   ⚠ auth_on_new_endpoints (0.93): Every new HTTP endpoint (a new @app route) must require authentication, e.g. with an @require_auth decorator
   ⚠ i18n_strings (0.81): User-facing strings (messages, titles or labels shown to users) must be wrapped in the t() translation function, e.g. t("orders.started")


DIFF_2: 0 comment(s)


DIFF_3: 2 comment(s)
   ⚠ auth_on_new_endpoints (0.90): Every new HTTP endpoint (a new @app route) must require authentication, e.g. with an @require_auth decorator
   ⚠ i18n_strings (0.82): User-facing strings (messages, titles or labels shown to users) must be wrapped in the t() translation function, e.g. t("orders.started")


DIFF_4: 0 comment(s)

rule fire counts over 4 diffs:
   no_raw_sql_in_handlers   1  (25%)
   no_logging_bodies        1  (25%)
   auth_on_new_endpoints    2  (50%)
   i18n_strings             2  (50%)


**What to notice**
- `DIFF_1` gets all four comments, as expected.
- `DIFF_3` is flagged for exactly the two real problems, **missing auth** on an admin endpoint and the untranslated title, and *not* for SQL, because it correctly uses `user_repo`.
- The two clean diffs, `DIFF_2` and `DIFF_4`, get **zero** comments. Silence on good code is what earns a bot the right to be listened to.
- In the fire counts, auth and i18n fire on half of this tiny batch because two of the four diffs really break them. On real traffic, a rule firing on a large share of PRs is your cue to look at its wording.

## Step 5 · Give it a face

Everything from here on is **presentation only**: a small style kit, a form wired to `review()` with `ipywidgets`, and
an HTML renderer that shows the diff (added lines in green) and one comment card per flagged rule. The review logic
above is untouched. Note that every render calls `review()`, so it also adds to `fired`.

In [8]:
import ipywidgets as widgets
from IPython.display import HTML, display

INK, SLATE, CORAL, SAGE, AMBER, RED, LINE = "#1F1E1D", "#5E6B7A", "#D97757", "#1F7A3D", "#C98A00", "#C0392B", "#DEDACE"
CARD = f"background:#FAF9F5;border:1px solid {LINE};border-radius:10px;padding:14px 16px;margin:8px 0;"
BADGE = "display:inline-block;padding:2px 10px;border-radius:12px;color:white;font-weight:600;font-size:12px;"


def box(inner):
    return f'<div style="font-family:-apple-system,Segoe UI,sans-serif;color:{INK};max-width:860px;">{inner}</div>'


def badge(text, color):
    return f'<span style="{BADGE}background:{color};">{text}</span>'


def meter(label, p, width=160):
    """A labelled horizontal probability bar."""
    col = RED if p > 0.7 else AMBER if p > 0.4 else SAGE
    return (f'<div style="display:flex;align-items:center;gap:8px;font-size:12px;margin:2px 0;">'
            f'<span style="width:190px;color:{SLATE};">{label}</span>'
            f'<span style="width:{width}px;height:8px;background:#ECE9E0;border-radius:4px;overflow:hidden;">'
            f'<span style="display:block;height:8px;width:{p * 100:.0f}%;background:{col};"></span></span>'
            f'<b>{p:.2f}</b></div>')


def form_app(fields, render, button="Run"):
    """fields: list of (name, label, default, rows). render(**values) returns HTML."""
    boxes = {}
    for name, label, default, rows in fields:
        w = (widgets.Textarea(value=default, rows=rows, layout=widgets.Layout(width="90%"))
             if rows > 1 else widgets.Text(value=default, layout=widgets.Layout(width="90%")))
        boxes[name] = w
    btn = widgets.Button(description=button, button_style="warning")
    out = widgets.Output()

    def go(_):
        with out:
            out.clear_output()
            display(HTML(box("<i>thinking...</i>")))
            html = render(**{k: w.value.strip() for k, w in boxes.items()})
            out.clear_output(wait=True)
            display(HTML(html))

    btn.on_click(go)
    rows = [widgets.VBox([widgets.HTML(f"<b>{label}</b>"), boxes[name]]) for name, label, _, _ in fields]
    display(widgets.VBox(rows + [btn]), out)

`review_html()` renders a diff and its comments. We show `DIFF_3`, the admin endpoint, as the example.

In [9]:
import html as htmllib


def review_html(diff):
    comments = review(diff, jev)
    lines = "".join(
        f'<div style="font-family:Menlo,monospace;font-size:12px;white-space:pre;'
        f'background:{"#E8F5E9" if ln.startswith("+") else "transparent"};">{htmllib.escape(ln)}</div>'
        for ln in diff.strip("\n").splitlines())
    notes = "".join(f'<div style="{CARD}padding:8px 12px;border-left:5px solid {AMBER};">'
                    f'{badge(f"{p:.2f}", AMBER)} <b>{k}</b><div style="font-size:12px;color:{SLATE};">{rule}</div></div>'
                    for k, p, rule in comments) or f'<div style="color:{SAGE};">✓ no rule violations flagged</div>'
    return box(f'<div style="{CARD}">{lines}</div>{notes}')


display(HTML(review_html(DIFF_3)))

## Step 6 · Try it yourself

Edit the diff and press **Review** (needs a live kernel). Keep a `+` at the start of each added line. Good ones to try:
- The default (`DELETE` endpoint with an f-string SQL query) → expect **raw SQL** and **auth** comments. The `"User deleted"` message usually scores mid-range on i18n, *below* the comment line: is a JSON `msg` user-facing? Reviewers would argue too, which makes it a good discussion point.
- Add a line `+@require_auth` under the route decorator → the **auth** comment should disappear.
- `+logger.info("user %s logged in", user.id)` → logs an id, not a body; expect **no** `no_logging_bodies` comment.
- `+    flash("Your password was changed")` → an untranslated message shown to a user; expect an **i18n** comment.

In [10]:
form_app([("diff", "Diff (added lines start with +)",
           '+@app.delete("/users/<id>")\n+def delete_user(request, id):\n+    db.execute(f"DELETE FROM users WHERE id={id}")\n+    return {"msg": "User deleted"}', 6)],
         review_html, "Review")

Output()

## Recap and exercises

**Key ideas**
1. **Rules in English, one `Noul` each.** The team's wiki page becomes an executable check, all rules in one call.
2. **`state` as a dict**: `` `diff` `` in the instructions points at the key, and "only the ADDED lines" scopes the judgment.
3. **Quiet beats clever.** Clean diffs must get zero comments; a per-rule **fire count** tells you which rule is noisy.
4. **Wording is tuning.** Concrete examples inside a rule moved real violations well clear of the threshold.

**Exercises**
- Add a rule `no_print_debugging` ("Don't leave print() calls in application code") and write a diff that breaks it.
- Raise `COMMENT_AT` to 0.85 and re-run Step 4. Which comments disappear? Are any of them ones you'd want to keep?
- Undo the Workshop-note wording (use the short versions of the auth and i18n rules) and compare the probabilities on `DIFF_1`.
- Give each rule its own threshold (a dict instead of one `COMMENT_AT`) so a noisy rule can be made stricter on its own.